# Smart City Parking System — Data Analysis
**DAV Project | Data Analysis Module**

This notebook analyses the cleaned parking dataset. Keep `Cleaned_Smart_Parking_Dataset.csv` in the same folder as this notebook before running it.

**Analysis goals**
- Summarize parking occupancy and parking-zone usage.
- Compare vehicle types, entry hours, parking duration, and payment status.
- Examine violations, EV records, and numeric relationships.
- Export summary tables for the Data Visualization team.

> This dataset contains 1,000 records. The results describe the supplied dataset; they should not be treated as live parking availability.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_FILE = Path("Cleaned_Smart_Parking_Dataset.csv")
OUTPUT_DIR = Path("analysis_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find {DATA_FILE}. Put the cleaned CSV in the same folder as this notebook."
    )

df = pd.read_csv(DATA_FILE)

# Convert timestamps safely when the column exists
if "Timestamp" in df.columns:
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())

## 1. Dataset overview and quality check

In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())

print("\\nData types and non-null counts:")
df.info()

quality_report = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(),
})
display(quality_report)
print("Duplicate rows:", int(df.duplicated().sum()))
quality_report.to_csv(OUTPUT_DIR / "data_quality_summary.csv", index=True)

## 2. Descriptive statistics

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns
numeric_summary = df[numeric_cols].describe().T.round(2)
display(numeric_summary)
numeric_summary.to_csv(OUTPUT_DIR / "numeric_summary.csv")

## 3. Occupancy analysis

In [ ]:
if "Occupancy_Status" in df.columns:
    occupancy_counts = df["Occupancy_Status"].value_counts(dropna=False).rename_axis("Occupancy_Status").reset_index(name="Records")
    occupancy_counts["Percent"] = (occupancy_counts["Records"] / len(df) * 100).round(2)
    display(occupancy_counts)
    occupancy_counts.to_csv(OUTPUT_DIR / "occupancy_status_summary.csv", index=False)

    if "Parking_Lot_Section" in df.columns:
        section_occupancy = pd.crosstab(
            df["Parking_Lot_Section"], df["Occupancy_Status"], margins=True
        )
        display(section_occupancy)
        section_occupancy.to_csv(OUTPUT_DIR / "section_occupancy_summary.csv")

if "Occupancy_Rate" in df.columns:
    print("Average Occupancy_Rate:", round(df["Occupancy_Rate"].mean(), 3))
    print("Median Occupancy_Rate:", round(df["Occupancy_Rate"].median(), 3))
    print("Minimum Occupancy_Rate:", round(df["Occupancy_Rate"].min(), 3))
    print("Maximum Occupancy_Rate:", round(df["Occupancy_Rate"].max(), 3))

## 4. Parking-zone and vehicle-type analysis

In [ ]:
for column in ["Parking_Lot_Section", "Vehicle_Type", "Spot_Size", "User_Type", "Nearby_Traffic_Level"]:
    if column in df.columns:
        summary = df[column].value_counts(dropna=False).rename_axis(column).reset_index(name="Records")
        summary["Percent"] = (summary["Records"] / len(df) * 100).round(2)
        print(f"\\n{column} distribution")
        display(summary)
        summary.to_csv(OUTPUT_DIR / f"{column.lower()}_distribution.csv", index=False)

if {"Vehicle_Type", "Parking_Duration"}.issubset(df.columns):
    duration_by_vehicle = df.groupby("Vehicle_Type")["Parking_Duration"].agg(
        Record_Count="count", Mean_Duration="mean", Median_Duration="median", Max_Duration="max"
    ).round(2)
    display(duration_by_vehicle)
    duration_by_vehicle.to_csv(OUTPUT_DIR / "parking_duration_by_vehicle.csv")

## 5. Entry-time and timestamp analysis

In [ ]:
if "Entry_Time" in df.columns:
    hourly_entries = df["Entry_Time"].value_counts().sort_index().rename_axis("Entry_Hour").reset_index(name="Records")
    display(hourly_entries)
    hourly_entries.to_csv(OUTPUT_DIR / "entry_hour_summary.csv", index=False)
    if not hourly_entries.empty:
        peak = hourly_entries.loc[hourly_entries["Records"].idxmax()]
        print(f"Most frequent entry hour in this dataset: {int(peak['Entry_Hour']):02d}:00 ({int(peak['Records'])} records)")

if "Timestamp" in df.columns:
    valid_timestamps = df["Timestamp"].dropna()
    if not valid_timestamps.empty:
        print("Timestamp range:", valid_timestamps.min(), "to", valid_timestamps.max())
        df["Month"] = df["Timestamp"].dt.to_period("M").astype(str)
        monthly_records = df.groupby("Month").size().rename("Records").reset_index()
        display(monthly_records.head(12))
        monthly_records.to_csv(OUTPUT_DIR / "monthly_record_counts.csv", index=False)

## 6. Payment analysis

In [ ]:
if "Payment_Status" in df.columns:
    payment_status = df["Payment_Status"].value_counts(dropna=False).rename_axis("Payment_Status").reset_index(name="Records")
    payment_status["Percent"] = (payment_status["Records"] / len(df) * 100).round(2)
    display(payment_status)
    payment_status.to_csv(OUTPUT_DIR / "payment_status_summary.csv", index=False)

if "Payment_Amount" in df.columns:
    print("Total recorded payment amount:", round(df["Payment_Amount"].sum(), 2))
    print("Average recorded payment amount:", round(df["Payment_Amount"].mean(), 2))
    print("Median recorded payment amount:", round(df["Payment_Amount"].median(), 2))

if {"Parking_Lot_Section", "Payment_Amount"}.issubset(df.columns):
    payment_by_section = df.groupby("Parking_Lot_Section")["Payment_Amount"].agg(
        Record_Count="count", Total_Amount="sum", Average_Amount="mean"
    ).round(2)
    display(payment_by_section)
    payment_by_section.to_csv(OUTPUT_DIR / "payment_by_section.csv")

## 7. Parking violations and electric vehicles

In [ ]:
if "Parking_Violation" in df.columns:
    print("Records flagged with Parking_Violation = 1:", int((df["Parking_Violation"] == 1).sum()))
    print("Violation record percentage:", round((df["Parking_Violation"] == 1).mean() * 100, 2), "%")
    if "Parking_Lot_Section" in df.columns:
        violations_by_section = df.groupby("Parking_Lot_Section")["Parking_Violation"].agg(
            Total_Records="count", Violation_Flags="sum", Violation_Rate="mean"
        )
        violations_by_section["Violation_Rate"] = (violations_by_section["Violation_Rate"] * 100).round(2)
        display(violations_by_section)
        violations_by_section.to_csv(OUTPUT_DIR / "violations_by_section.csv")

if "Electric_Vehicle" in df.columns:
    ev_summary = df["Electric_Vehicle"].value_counts(dropna=False).rename_axis("Electric_Vehicle (1=yes, 0=no)").reset_index(name="Records")
    ev_summary["Percent"] = (ev_summary["Records"] / len(df) * 100).round(2)
    display(ev_summary)
    ev_summary.to_csv(OUTPUT_DIR / "electric_vehicle_summary.csv")

## 8. Numeric relationships (correlation)

In [ ]:
numeric_df = df.select_dtypes(include="number")
if numeric_df.shape[1] >= 2:
    correlation = numeric_df.corr().round(2)
    display(correlation)
    correlation.to_csv(OUTPUT_DIR / "numeric_correlation_matrix.csv")
else:
    print("Not enough numeric columns for correlation analysis.")

## 9. Export a compact findings summary

In [ ]:
findings = []
findings.append(f"Dataset contains {len(df)} records and {df.shape[1]} columns.")
findings.append(f"Total missing cells: {int(df.isna().sum().sum())}.")
findings.append(f"Duplicate rows: {int(df.duplicated().sum())}.")

if "Occupancy_Status" in df.columns:
    counts = df["Occupancy_Status"].value_counts()
    for status, count in counts.items():
        findings.append(f"{status} records: {int(count)} ({count / len(df) * 100:.2f}%).")

if "Parking_Duration" in df.columns:
    findings.append(f"Average parking duration: {df['Parking_Duration'].mean():.2f} dataset units.")

if "Payment_Amount" in df.columns:
    findings.append(f"Average recorded payment amount: {df['Payment_Amount'].mean():.2f} dataset currency units.")

if "Parking_Violation" in df.columns:
    findings.append(f"Records with violation flag = 1: {int((df['Parking_Violation'] == 1).sum())}.")

findings_df = pd.DataFrame({"Finding": findings})
display(findings_df)
findings_df.to_csv(OUTPUT_DIR / "key_findings.csv", index=False)

print(f"\\nAnalysis complete. Output files saved in: {OUTPUT_DIR.resolve()}")